# 01 · Resumen de la sonda del Tema 1

Este notebook resume en una sola página la **sonda** del Tema 1 (mercados y series: precios, curvas, divisas, series
macro con vintages, libre de riesgo, baselines y costes; tools `get_serie(id, fecha)` y `get_precios(tk, fecha)`), que
antes ocupaba nueve notebooks. Es el paso 1 de la descarga completa; el siguiente es `02_inventario`
([plan de la descarga](../../../docs/fase2/mercados_series/PLAN_DESCARGA.md)).

Las cifras del corpus salen de las celdas de código; el texto no lleva ninguna escrita a mano. El detalle de cada bloque
está en el [README del tema](../../../src/ingesta/mercados_series/README.md).

## Lo que hizo la sonda, notebook a notebook

| Notebook | Qué hizo | Qué dejó |
|---|---|---|
| `01_contrato_timegate` | Fijó el contrato (tablas de series y de precios) y el reloj de decisión, y probó el truncado en directo con datos sintéticos, sin red | Módulos `contrato`, `visibilidad`, `timegate`, `tools` y `sinteticos`; truncado sin diferencias |
| `02_base_fase1` | Unió `data/raw` de la Fase 1, los sustitutos de FRED y las académicas (French, Shiller, Goyal-Welch, OFR FSI) con fecha de publicación en cada fila | `series/base_series.parquet` y `construir-base`; fugas de los cierres del Gobierno, Moody's, H.15 y LIBOR corregidas con ALFRED |
| `03_alfred_vintages` | Cargó las revisables de FRED con todas sus vintages de ALFRED y midió revisiones, retrasos e índices re-estimados (NFCI) | Vintages exactos en la base (`descargar-alfred`), ediciones archivadas de las académicas y aviso «re-estimado entero» |
| `04_macro_internacional` | Añadió BCE, Eurostat, BIS, FMI, Banco Mundial, OCDE, Philadelphia Fed y bancos centrales, y las curvas oficiales (Tesoro, BCE, MoF, BoC, BoE) | Cargadores `base.tabla_oficial`, prioridad de fuentes al unir, empalme ECOICOP v2 del HICP y alias `HICP_EA`, `PIB_EA`, `PARO_EA` |
| `05_universo` | Validó el universo de instrumentos (ETF, baselines, UCITS, índices, divisas, futuros y cripto) y la composición point-in-time del S&P 500 | `config/universo.yaml` validado (ISIN y UCITS verificados) y `constituyentes/sp500.parquet` |
| `06_precios_cobertura` | Descargó de Yahoo el núcleo y el nivel ampliado as-traded y midió inicio, cobertura, calidad y retraso por mercado | `precios/yahoo/`, `timegate.des_ajustar_yahoo` y `ajustar_a_fecha`, `calidad.incidencias_precios`; fuera de EE. UU. Yahoo tiene poca historia |
| `07_supervivencia` | Midió el sesgo de supervivencia del S&P 500: precio de los salidos, tickers reutilizados y cartera de supervivientes frente a point-in-time | Ficha `supervivencia` y `tablas/supervivencia_cobertura.csv`; la selección de acciones con Yahoo no vale sin corregir, los ETF e índices sí |
| `08_libre_riesgo_baselines_costes` | Libre de riesgo visible (`fred:DTB3`; EONIA y luego €STR en EUR), baselines solo con `get_precios` y costes con multiplicador de crisis | Paquete `cartera` (`cartera.informe`), curvas en `<corpus>/cartera/` y `tablas/costes_modelo.csv` |
| `09_cierre_calidad` | Juntó fichas, cobertura, huecos, anomalías y retrasos, y probó las dos tools en la quiebra de Lehman y el lunes del COVID | Estado de cada ficha, `tablas/cobertura_series.csv` y `anomalias_series.csv`, y el test `test_t1_cierre.py` |

## Decisiones del contrato que siguen vigentes

- **Hora de decisión:** 16:30 de Nueva York (con cambio de hora) de cada día natural. Un cierre es visible a su hora
  local de cierre + 30 min; una publicación, a su hora declarada (`config/mercados.yaml`, `config/publicaciones.yaml`).
- **TimeGate:** `fecha_publicacion` es la primera decisión posterior a `disponible_utc`; las tools solo devuelven filas
  con `fecha_publicacion <= fecha` y la base solo crece, así que la respuesta en una fecha no cambia al añadir datos
  posteriores (truncado). Sin nada publicado, la tool responde como si el id no existiera.
- **Precios as-traded:** nunca se guarda `Adj Close`; los splits de Yahoo se deshacen y el ajuste se reconstruye a la
  fecha con los eventos visibles entonces.
- **USD al leer:** se guarda la divisa original y se convierte con el FX de referencia del BCE visible en la fecha.
- **Vintages:** las revisables van con sus vintages de ALFRED (o ediciones `AAAA-MM`), nunca con el histórico de hoy.
- **Ids `fuente:ID`** (`fred:DGS10`, `ecb:EXR.D.USD.EUR`), con alias opcionales; huecos permanentes como nulo con nota.
- **Alcance:** EE. UU., zona euro, G7 y China.
- **Datos fuera del repo:** corpus en `D:/tfm_corpus/mercados_series` (`INGESTA_CORPUS_DIR` en `.env`); en el repo solo
  fichas y tablas de metadatos (`results/ingesta/mercados_series/`).

**Deudas principales** (lista completa en el [README, «Deudas pendientes»](../../../src/ingesta/mercados_series/README.md#deudas-pendientes)):

- Supervivencia: sin WRDS/CRSP, los deslistados del S&P 500 siguen sin precio y el sesgo medido es una cota inferior.
- Horas históricas sin verificar: H.15 diario antiguo y reglas de Ken French antes de sus ediciones archivadas.
- FMI WEO sin ediciones históricas (DBnomics lo veta), BoE IADB vetado y BCE, Eurostat, BIS y Banco Mundial sin
  historia de revisiones.
- Archivo de las OAS de ICE y de las ediciones académicas, a mano; comisiones sin verificar; horizontes del SEP sin
  decidir.

## El corpus hoy

Dos celdas: la base de series por fuente (solo tres columnas de la base) y los precios por proveedor con el estado de
las fichas.

In [1]:
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

from ingesta.comun.rutas import RESULTS_INGESTA, corpus_dir

try:
    CORPUS = corpus_dir('mercados_series')
except OSError as e:  # unidad del corpus no montada
    CORPUS = None
    print(f'Corpus no disponible ({e}).')
BASE = CORPUS / 'series' / 'base_series.parquet' if CORPUS else None

if BASE is None or not BASE.exists():
    print(f'No hay base de series en {BASE}: se omiten sus cifras.')
    POR_FUENTE = None
else:
    tabla = pq.read_table(BASE, columns=['serie', 'fuente', 'fecha_referencia'], read_dictionary=['serie', 'fuente'])
    df = tabla.to_pandas()
    POR_FUENTE = (df.groupby('fuente', observed=True)
                    .agg(filas=('serie', 'size'), series=('serie', 'nunique'),
                         desde=('fecha_referencia', 'min'), hasta=('fecha_referencia', 'max'))
                    .sort_values('filas', ascending=False))
    POR_FUENTE[['desde', 'hasta']] = POR_FUENTE[['desde', 'hasta']].apply(lambda c: c.dt.date)
    print(f"Base: {len(df):,} filas · {df['serie'].nunique():,} series · {len(POR_FUENTE)} fuentes · "
          f"referencias de {df['fecha_referencia'].min():%Y-%m-%d} a {df['fecha_referencia'].max():%Y-%m-%d} · "
          f"{BASE.stat().st_size / 1e6:,.0f} MB")
    del df, tabla
POR_FUENTE

Base: 10,516,135 filas · 1,434 series · 24 fuentes · referencias de 1854-12-01 a 2029-01-01 · 96 MB


,filas,series,desde,hasta
fuente,,,,
fred_alfred,7815910,241,1854-12-01,2029-01-01
oecd,953171,26,1949-01-01,2026-09-01
ken_french,374353,17,1926-07-01,2026-08-31
yfinance,276787,26,1927-12-30,2026-07-17
bce,272188,52,1995-01-01,2026-10-12
mof_jgb,165042,15,1974-09-24,2026-10-09
bce_curvas,124300,22,2004-09-06,2026-10-12
boe_curvas,122752,12,1979-01-02,2026-10-12
calculo_propio,103216,9,1928-01-16,2026-10-08


In [2]:
PRECIOS = CORPUS / 'precios' if CORPUS else None
if PRECIOS is None or not PRECIOS.exists():
    print(f'No hay carpeta de precios en {PRECIOS}.')
else:
    n = {p.name: sum(1 for _ in p.glob('*.parquet')) for p in sorted(PRECIOS.iterdir()) if p.is_dir()}
    print('Ficheros de precios por proveedor: ' + ' · '.join(f'{k} {v:,}' for k, v in n.items())
          + f' · total {sum(n.values()):,}')
if CORPUS is not None and CORPUS.exists():
    gb = sum(p.stat().st_size for p in CORPUS.rglob('*') if p.is_file()) / 1e9
    print(f'Corpus entero en disco: {gb:,.1f} GB')

FICHAS_DIR = RESULTS_INGESTA / 'mercados_series' / 'fichas'
fichas = [json.loads(p.read_text(encoding='utf-8')) for p in sorted(FICHAS_DIR.glob('*.json'))]
if not fichas:
    print(f'No hay fichas en {FICHAS_DIR}.')
else:
    TF = pd.DataFrame([{'ficha': f['fuente'], 'estado': f.get('estado_sonda'), 'fecha_sonda': f.get('fecha_sonda')}
                       for f in fichas])
    print(f'{len(TF)} fichas · última sonda {TF["fecha_sonda"].max()}')
    for estado, g in TF.groupby('estado'):
        print(f'  {estado}: {len(g)}' + ('' if estado == 'ok' else ' → ' + ', '.join(g['ficha'])))

Ficheros de precios por proveedor: coinbase 2 · yahoo 1,469 · total 1,471


Corpus entero en disco: 2.5 GB
34 fichas · última sonda 2026-10-14
  fallo: 1 → boe
  ok: 29
  parcial: 4 → ice_bofa_oas, imf_weo, indices_constituyentes, oecd


## Qué tenemos para empezar el paso 2

- Una base de series con fecha de publicación en cada fila y vintages donde hay revisiones, leída por `get_serie`
  con TimeGate y truncado probados (tabla por fuente de arriba).
- Un almacén de precios as-traded por proveedor, leído por `get_precios` con el ajuste a fecha (recuento de arriba).
- Una ficha por fuente con su estado, cobertura y retraso medidos: las que no están en `ok` marcan dónde faltan
  datos (lista de arriba).
- El criterio para quedarse con un dato (uso, información nueva, fecha fiable, historia) y las decisiones pendientes,
  en el [plan de la descarga](../../../docs/fase2/mercados_series/PLAN_DESCARGA.md): `02_inventario` las aplica a este corpus.

<small>Los nueve notebooks originales se conservan en la etiqueta git `sonda-tema1`.</small>